# Asteroid Risk Prediction using NASA Dataset
**Group:** M Haseeb (22L-6577) | Aun Noman (22L-6950)  
**Course:** Data Science — BCS8A

## 1. Setup & Data Loading

In [42]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc
)

sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
print('Libraries loaded successfully.')

Libraries loaded successfully.


In [43]:
df = pd.read_csv('./dataset.csv', low_memory=False)
print(f'Dataset shape: {df.shape}')
print(f'\nColumn list:')
print(df.columns.tolist())

Dataset shape: (291764, 45)

Column list:
['id', 'spkid', 'full_name', 'pdes', 'name', 'prefix', 'neo', 'pha', 'H', 'diameter', 'albedo', 'diameter_sigma', 'orbit_id', 'epoch', 'epoch_mjd', 'epoch_cal', 'equinox', 'e', 'a', 'q', 'i', 'om', 'w', 'ma', 'ad', 'n', 'tp', 'tp_cal', 'per', 'per_y', 'moid', 'moid_ld', 'sigma_e', 'sigma_a', 'sigma_q', 'sigma_i', 'sigma_om', 'sigma_w', 'sigma_ma', 'sigma_ad', 'sigma_n', 'sigma_tp', 'sigma_per', 'class', 'rms']


In [44]:
df.head(5)

,id,spkid,full_name,pdes,name,prefix,neo,pha,H,diameter,...,sigma_i,sigma_om,sigma_w,sigma_ma,sigma_ad,sigma_n,sigma_tp,sigma_per,class,rms
0,a0000001,2000001,1 Ceres,1,Ceres,NaN,N,N,3.40,939.400,...,4.608900e-09,6.168800e-08,6.624800e-08,7.820700e-09,1.111300e-11,1.196500e-12,3.782900e-08,9.415900e-09,MBA,0.43301
1,a0000002,2000002,2 Pallas,2,Pallas,NaN,N,N,4.20,545.000,...,3.469400e-06,6.272400e-06,9.128200e-06,8.859100e-06,4.961300e-09,4.653600e-10,4.078700e-05,3.680700e-06,MBA,0.35936
2,a0000003,2000003,3 Juno,3,Juno,NaN,N,N,5.33,246.596,...,3.223100e-06,1.664600e-05,1.772100e-05,8.110400e-06,4.363900e-09,4.413400e-10,3.528800e-05,3.107200e-06,MBA,0.33848
3,a0000004,2000004,4 Vesta,4,Vesta,NaN,N,N,3.00,525.400,...,2.170600e-07,3.880800e-07,1.789300e-07,1.206800e-06,1.648600e-09,2.612500e-10,4.103700e-06,1.274900e-06,MBA,0.39980
4,a0000005,2000005,5 Astraea,5,Astraea,NaN,N,N,6.90,106.699,...,2.740800e-06,2.894900e-05,2.984200e-05,8.303800e-06,4.729000e-09,5.522700e-10,3.474300e-05,3.490500e-06,MBA,0.52191


In [45]:
print('Data types and non-null counts:')
df.info()

Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 291764 entries, 0 to 291763
Data columns (total 45 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              291764 non-null  object 
 1   spkid           291764 non-null  int64  
 2   full_name       291764 non-null  object 
 3   pdes            291764 non-null  int64  
 4   name            21537 non-null   object 
 5   prefix          0 non-null       float64
 6   neo             291764 non-null  object 
 7   pha             291764 non-null  object 
 8   H               291764 non-null  float64
 9   diameter        106689 non-null  float64
 10  albedo          106651 non-null  float64
 11  diameter_sigma  106572 non-null  float64
 12  orbit_id        291764 non-null  object 
 13  epoch           291764 non-null  float64
 14  epoch_mjd       291764 non-null  int64  
 15  epoch_cal       291764 non-null  float64
 16  equinox         291764 n

## 2. Exploratory Data Analysis (EDA)

### 2.1 Target Variable — Hazard Status

In [46]:
# Drop rows where pha is null
pha_counts = df['pha'].value_counts(dropna=False)
print('PHA value counts (including NaN):')
print(pha_counts)

df_eda = df.dropna(subset=['pha']).copy()
df_eda['pha_binary'] = (df_eda['pha'].str.strip().str.upper() == 'Y').astype(int)

print(f'\nAfter dropping null pha: {df_eda.shape[0]} rows')
print(f"Class distribution:\n{df_eda['pha_binary'].value_counts()}")
print(f"Hazardous proportion: {df_eda['pha_binary'].mean()*100:.2f}%")

PHA value counts (including NaN):
pha
N    291444
Y       320
Name: count, dtype: int64

After dropping null pha: 291764 rows
Class distribution:
pha_binary
0    291444
1       320
Name: count, dtype: int64
Hazardous proportion: 0.11%


In [47]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df_eda['pha_binary'].value_counts()
axes[0].bar(['Non-Hazardous (0)', 'Hazardous (1)'], counts.values,
            color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Class Distribution (Absolute)')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Non-Hazardous', 'Hazardous'],
            autopct='%1.2f%%', colors=['steelblue', 'tomato'],
            startangle=90, explode=(0, 0.05))
axes[1].set_title('Class Distribution (Proportion)')

plt.tight_layout()
plt.savefig('./fig_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


### 2.2 Missing Value Analysis

In [48]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print('Columns with missing values:')
print(missing_df)

Columns with missing values:
                Missing Count  Missing %
prefix                 291764     100.00
name                   270227      92.62
diameter_sigma         185192      63.47
albedo                 185113      63.45
diameter               185075      63.43
tp_cal                      1       0.00
per                         1       0.00
per_y                       1       0.00
moid                        1       0.00
moid_ld                     1       0.00
sigma_e                     2       0.00
sigma_a                     2       0.00
sigma_q                     2       0.00
sigma_i                     2       0.00
sigma_om                    2       0.00
sigma_w                     2       0.00
sigma_ma                    2       0.00
sigma_ad                    2       0.00
sigma_n                     2       0.00
sigma_tp                    2       0.00
sigma_per                   2       0.00
class                       1       0.00
rms                         

In [49]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(missing_df.index, missing_df['Missing %'], color='salmon', edgecolor='black')
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Value Percentage by Column')
ax.axvline(x=50, color='red', linestyle='--', label='50% threshold')
ax.legend()
plt.tight_layout()
plt.savefig('./fig_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Univariate Analysis — Feature Distributions

In [50]:
NUMERIC_FEATURES = ['H', 'diameter', 'albedo', 'e', 'a', 'q', 'i', 'moid', 'n', 'per_y']
existing_features = [f for f in NUMERIC_FEATURES if f in df_eda.columns]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for idx, feat in enumerate(existing_features):
    data = df_eda[feat].dropna()
    # Clip at 1st and 99th percentile for readability
    p1, p99 = data.quantile(0.01), data.quantile(0.99)
    data_clipped = data.clip(p1, p99)
    axes[idx].hist(data_clipped, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[idx].set_title(feat)
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')

# Hide unused axes
for idx in range(len(existing_features), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Univariate Feature Distributions (clipped at 1st–99th percentile)', fontsize=14)
plt.tight_layout()
plt.savefig('./fig_univariate.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.4 Bivariate Analysis — Features vs Hazard Status

In [51]:
KEY_FEATURES = ['H', 'diameter', 'e', 'moid', 'a', 'i']
key_existing = [f for f in KEY_FEATURES if f in df_eda.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

labels = {0: 'Non-Hazardous', 1: 'Hazardous'}
colors = {0: 'steelblue', 1: 'tomato'}

for idx, feat in enumerate(key_existing):
    ax = axes[idx]
    data = []
    tick_labels = []
    for cls in [0, 1]:
        vals = df_eda[df_eda['pha_binary'] == cls][feat].dropna()
        p1, p99 = vals.quantile(0.01), vals.quantile(0.99)
        data.append(vals.clip(p1, p99))
        tick_labels.append(labels[cls])
    bp = ax.boxplot(data, patch_artist=True, notch=False)
    for patch, cls in zip(bp['boxes'], [0, 1]):
        patch.set_facecolor(colors[cls])
        patch.set_alpha(0.7)
    ax.set_xticklabels(tick_labels)
    ax.set_title(f'{feat} by Hazard Status')
    ax.set_ylabel(feat)

for idx in range(len(key_existing), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Bivariate Analysis: Key Features vs Hazard Status', fontsize=14)
plt.tight_layout()
plt.savefig('./fig_bivariate.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.5 Correlation Heatmap

In [52]:
corr_cols = [f for f in existing_features if f in df_eda.columns] + ['pha_binary']
corr_data = df_eda[corr_cols].copy()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('./fig_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.6 Outlier Detection

In [ ]:
def count_outliers_iqr(series):
    series = series.dropna()
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((series < lower) | (series > upper)).sum()
    return n_out, n_out / len(series) * 100

outlier_report = []
for feat in existing_features:
    if feat in df_eda.columns:
        n, pct = count_outliers_iqr(df_eda[feat])
        outlier_report.append({'Feature': feat, 'Outlier Count': n, 'Outlier %': round(pct, 2)})

outlier_df = pd.DataFrame(outlier_report).sort_values('Outlier %', ascending=False)
print('IQR-based outlier detection:')
print(outlier_df.to_string(index=False))

IQR-based outlier detection:
 Feature  Outlier Count  Outlier %
diameter           7636       7.16
       i          10392       3.56
       H           6479       2.22
       a           4401       1.51
   per_y           4269       1.46
       n           4213       1.44
       q           3786       1.30
  albedo           1323       1.24
    moid           3511       1.20
       e           2577       0.88


In [54]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(outlier_df['Feature'], outlier_df['Outlier %'], color='orange', edgecolor='black')
ax.set_xlabel('Outlier %')
ax.set_title('IQR-Based Outlier Percentage by Feature')
plt.tight_layout()
plt.savefig('./fig_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Data Preprocessing

In [55]:
# Start fresh from raw data
df_clean = df.dropna(subset=['pha']).copy()
df_clean['pha_binary'] = (df_clean['pha'].str.strip().str.upper() == 'Y').astype(int)

# Drop non-predictive / identifier / date columns
DROP_COLS = [
    'id', 'spkid', 'full_name', 'pdes', 'name', 'prefix',
    'tp_cal', 'epoch_cal', 'equinox', 'pha', 'neo',
    'class', 'orbit_id', 'epoch'
]
drop_existing = [c for c in DROP_COLS if c in df_clean.columns]
df_clean.drop(columns=drop_existing, inplace=True)
print(f'After dropping identifier columns: {df_clean.shape}')

# Drop columns with >50% missing values
missing_frac = df_clean.isnull().mean()
high_missing = missing_frac[missing_frac > 0.5].index.tolist()
print(f'Columns with >50% missing (dropped): {high_missing}')
df_clean.drop(columns=high_missing, inplace=True)

# Drop remaining string columns
str_cols = df_clean.select_dtypes(include='object').columns.tolist()
print(f'Remaining string columns (dropped): {str_cols}')
df_clean.drop(columns=str_cols, inplace=True)

print(f'\nFinal feature set shape: {df_clean.shape}')
print(f'Features: {[c for c in df_clean.columns if c != "pha_binary"]}')

After dropping identifier columns: (291764, 32)
Columns with >50% missing (dropped): ['diameter', 'albedo', 'diameter_sigma']
Remaining string columns (dropped): []

Final feature set shape: (291764, 29)
Features: ['H', 'epoch_mjd', 'e', 'a', 'q', 'i', 'om', 'w', 'ma', 'ad', 'n', 'tp', 'per', 'per_y', 'moid', 'moid_ld', 'sigma_e', 'sigma_a', 'sigma_q', 'sigma_i', 'sigma_om', 'sigma_w', 'sigma_ma', 'sigma_ad', 'sigma_n', 'sigma_tp', 'sigma_per', 'rms']


In [56]:
# Separate features and target
X = df_clean.drop(columns=['pha_binary'])
y = df_clean['pha_binary']

print(f'Feature matrix: {X.shape}')
print(f'Target vector : {y.shape}')
print(f'Class balance : {y.value_counts().to_dict()}')

Feature matrix: (291764, 28)
Target vector : (291764,)
Class balance : {0: 291444, 1: 320}


In [57]:
# Impute missing values with median
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
print(f'Missing values after imputation: {X_imputed.isnull().sum().sum()}')

Missing values after imputation: 0


In [58]:
# Train / test split — 70% / 30%, stratified
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training set : {X_train_sc.shape}')
print(f'Test set     : {X_test_sc.shape}')
print(f'Train hazardous %: {y_train.mean()*100:.2f}%')
print(f'Test  hazardous %: {y_test.mean()*100:.2f}%')

Training set : (204234, 28)
Test set     : (87530, 28)
Train hazardous %: 0.11%
Test  hazardous %: 0.11%


In [59]:
# Stratified subsample for SVM and KNN (50k rows)
SAMPLE_SIZE = 50_000
idx_sample = []
for cls in [0, 1]:
    cls_idx = np.where(y_train == cls)[0]
    n = min(int(SAMPLE_SIZE * y_train.mean()) if cls == 1 else int(SAMPLE_SIZE * (1 - y_train.mean())), len(cls_idx))
    idx_sample.extend(np.random.default_rng(RANDOM_STATE).choice(cls_idx, size=n, replace=False))

X_train_sample = X_train_sc[idx_sample]
y_train_sample = y_train.iloc[idx_sample]
print(f'SVM/KNN training sample: {X_train_sample.shape}')
print(f'Sample hazardous %: {y_train_sample.mean()*100:.2f}%')

SVM/KNN training sample: (49999, 28)
Sample hazardous %: 0.11%


## 4. Model Training

### 4.1 Logistic Regression (Baseline)

In [60]:
print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)
lr.fit(X_train_sc, y_train)
print('Done.')

Training Logistic Regression...
Done.


### 4.2 Random Forest

In [61]:
print('Training Random Forest (100 estimators)...')
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_sc, y_train)
print('Done.')

Training Random Forest (100 estimators)...
Done.


### 4.3 Support Vector Machine

In [62]:
print(f'Training SVM on {len(X_train_sample):,}-row sample...')
svm = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)
svm.fit(X_train_sample, y_train_sample)
print('Done.')

Training SVM on 49,999-row sample...
Done.


### 4.4 K-Nearest Neighbors

In [63]:
print(f'Training KNN (k=5) on {len(X_train_sample):,}-row sample...')
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn.fit(X_train_sample, y_train_sample)
print('Done.')

Training KNN (k=5) on 49,999-row sample...
Done.


## 5. Model Evaluation

In [64]:
models = {
    'Logistic Regression': lr,
    'Random Forest':       rf,
    'SVM':                 svm,
    'KNN':                 knn,
}

results = {}

for name, model in models.items():
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    results[name] = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred, zero_division=0),
        'f1':        f1_score(y_test, y_pred, zero_division=0),
        'auc':       roc_auc,
        'fpr':       fpr,
        'tpr':       tpr,
        'y_pred':    y_pred,
    }
    print(f'{name}: Acc={results[name]["accuracy"]:.4f}  '
          f'Prec={results[name]["precision"]:.4f}  '
          f'Rec={results[name]["recall"]:.4f}  '
          f'F1={results[name]["f1"]:.4f}  '
          f'AUC={results[name]["auc"]:.4f}')

Logistic Regression: Acc=0.9994  Prec=0.7531  Rec=0.6354  F1=0.6893  AUC=0.9998
Random Forest: Acc=1.0000  Prec=0.9897  Rec=1.0000  F1=0.9948  AUC=1.0000
SVM: Acc=0.9989  Prec=0.5556  Rec=0.2083  F1=0.3030  AUC=0.9963
KNN: Acc=0.9989  Prec=0.5000  Rec=0.2500  F1=0.3333  AUC=0.9160


### 5.1 Summary Comparison Table

In [65]:
summary = pd.DataFrame({
    name: {
        'Accuracy':  round(v['accuracy'],  4),
        'Precision': round(v['precision'], 4),
        'Recall':    round(v['recall'],    4),
        'F1-Score':  round(v['f1'],        4),
        'AUC':       round(v['auc'],       4),
    }
    for name, v in results.items()
}).T

print('Model Performance Summary:')
print(summary.to_string())

Model Performance Summary:
                     Accuracy  Precision  Recall  F1-Score     AUC
Logistic Regression    0.9994     0.7531  0.6354    0.6893  0.9998
Random Forest          1.0000     0.9897  1.0000    0.9948  1.0000
SVM                    0.9989     0.5556  0.2083    0.3030  0.9963
KNN                    0.9989     0.5000  0.2500    0.3333  0.9160


In [66]:
# Save summary to CSV for the report
summary.to_csv('./model_results.csv')
print('Results saved to model_results.csv')

Results saved to model_results.csv


### 5.2 Confusion Matrices

In [67]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, (name, v) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, v['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non-Haz', 'Haz'],
                yticklabels=['Non-Haz', 'Haz'],
                cbar=False)
    ax.set_title(f'{name}\nAcc={v["accuracy"]:.3f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All Models', fontsize=14)
plt.tight_layout()
plt.savefig('./fig_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Classification Reports

In [68]:
for name, v in results.items():
    print(f'\n--- {name} ---')
    print(classification_report(y_test, v['y_pred'],
                                target_names=['Non-Hazardous', 'Hazardous']))


--- Logistic Regression ---
               precision    recall  f1-score   support

Non-Hazardous       1.00      1.00      1.00     87434
    Hazardous       0.75      0.64      0.69        96

     accuracy                           1.00     87530
    macro avg       0.88      0.82      0.84     87530
 weighted avg       1.00      1.00      1.00     87530


--- Random Forest ---
               precision    recall  f1-score   support

Non-Hazardous       1.00      1.00      1.00     87434
    Hazardous       0.99      1.00      0.99        96

     accuracy                           1.00     87530
    macro avg       0.99      1.00      1.00     87530
 weighted avg       1.00      1.00      1.00     87530


--- SVM ---
               precision    recall  f1-score   support

Non-Hazardous       1.00      1.00      1.00     87434
    Hazardous       0.56      0.21      0.30        96

     accuracy                           1.00     87530
    macro avg       0.78      0.60      0.65   

### 5.4 ROC Curves

In [69]:
fig, ax = plt.subplots(figsize=(9, 7))

colors = ['steelblue', 'forestgreen', 'tomato', 'darkorange']
for (name, v), color in zip(results.items(), colors):
    ax.plot(v['fpr'], v['tpr'], color=color, lw=2,
            label=f'{name} (AUC = {v["auc"]:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./fig_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Metric Bar Chart Comparison

In [70]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']
model_names = list(results.keys())
x = np.arange(len(metrics))
width = 0.2
bar_colors = ['steelblue', 'forestgreen', 'tomato', 'darkorange']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (name, color) in enumerate(zip(model_names, bar_colors)):
    vals = [summary.loc[name, m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=name, color=color, alpha=0.85, edgecolor='black')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('./fig_metric_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Feature Importance

### 6.1 Random Forest Feature Importances

In [71]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(importances_sorted) * 0.35)))
colors_imp = ['forestgreen' if v >= importances.quantile(0.75) else 'steelblue'
              for v in importances_sorted.values]
importances_sorted.plot(kind='barh', ax=ax, color=colors_imp, edgecolor='black', alpha=0.85)
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('Random Forest — Feature Importances\n(green = top 25%)', fontsize=13)
ax.axvline(x=importances.mean(), color='red', linestyle='--', label='Mean importance')
ax.legend()
plt.tight_layout()
plt.savefig('./fig_rf_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 most important features:')
print(importances.sort_values(ascending=False).head(10).to_string())


Top 10 most important features:
moid_ld     0.388335
moid        0.326490
q           0.070650
per         0.025761
sigma_q     0.024947
per_y       0.023955
H           0.016470
a           0.016287
sigma_tp    0.014144
i           0.012257


### 6.2 Logistic Regression Coefficient Magnitudes

In [72]:
coef = pd.Series(np.abs(lr.coef_[0]), index=X.columns)
coef_sorted = coef.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(coef_sorted) * 0.35)))
coef_sorted.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black', alpha=0.85)
ax.set_xlabel('|Coefficient|', fontsize=12)
ax.set_title('Logistic Regression — Coefficient Magnitudes', fontsize=13)
plt.tight_layout()
plt.savefig('./fig_lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary & Conclusions

In [73]:
best_f1 = summary['F1-Score'].idxmax()
best_recall = summary['Recall'].idxmax()
best_auc = summary['AUC'].idxmax()

print('=== Final Results Summary ===')
print(summary.to_string())
print(f'\nBest F1-Score  : {best_f1} ({summary.loc[best_f1, "F1-Score"]:.4f})')
print(f'Best Recall    : {best_recall} ({summary.loc[best_recall, "Recall"]:.4f})')
print(f'Best AUC       : {best_auc} ({summary.loc[best_auc, "AUC"]:.4f})')
print(f'\nFor hazard detection, Recall is the most critical metric.')
print(f'Recommended model: {best_recall} (highest recall = fewest missed hazards)')

=== Final Results Summary ===
                     Accuracy  Precision  Recall  F1-Score     AUC
Logistic Regression    0.9994     0.7531  0.6354    0.6893  0.9998
Random Forest          1.0000     0.9897  1.0000    0.9948  1.0000
SVM                    0.9989     0.5556  0.2083    0.3030  0.9963
KNN                    0.9989     0.5000  0.2500    0.3333  0.9160

Best F1-Score  : Random Forest (0.9948)
Best Recall    : Random Forest (1.0000)
Best AUC       : Random Forest (1.0000)

For hazard detection, Recall is the most critical metric.
Recommended model: Random Forest (highest recall = fewest missed hazards)
